In [0]:
from pyspark.sql.functions import *
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType, LongType, DecimalType, TimestampType
from pyspark.sql.window import Window

In [0]:
# dbutils.widgets.text("delimitador",",")
# dbutils.widgets.text("dir_adls","abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/ordenes/oap/det_numero_portados_in") 
# dbutils.widgets.text("ruta_stage","abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/ordenes/oap/det_numero_portados_in/stage")
# dbutils.widgets.text("nombre_archivo","DetallesNumerosPortados_20250214.csv")
# dbutils.widgets.text("encabezado","false")
# dbutils.widgets.text("encoding","UTF-8")
# dbutils.widgets.text("quote","\"")
# dbutils.widgets.text("catalogo","bi_ingestas")
# dbutils.widgets.text("table_name","det_numero_portados_in")
# dbutils.widgets.text("dateFormatFile","yyyyMMdd")
# dbutils.widgets.text("timestampFormat","MM-dd-yyyy HH:mm:ss")
# dbutils.widgets.text("partition_date_column","hora_envio_solicitud")
# dbutils.widgets.text("schema","raw_interacciones")
# dbutils.widgets.text("original_file_date","2025-02-18T17:07:39Z")
# dbutils.widgets.text("starttime_nifi","20250218 143056")
# dbutils.widgets.text("endtime_nifi","20250218 143818")
# dbutils.widgets.text("original_file_size","366411")
# dbutils.widgets.text("pipelineRunId","sadasdasda")
# dbutils.widgets.text("catalog_control","bi_ingestas.control.control_ingestas")

In [0]:
delimitador     = dbutils.widgets.get("delimitador")
dir_adls        = dbutils.widgets.get("dir_adls") 
ruta_stage      = dbutils.widgets.get("ruta_stage")
nombre_archivo  = dbutils.widgets.get("nombre_archivo")
encabezado      = dbutils.widgets.get("encabezado").lower()
encoding        = dbutils.widgets.get("encoding")
quote           = dbutils.widgets.get("quote")
catalogo        = dbutils.widgets.get("catalogo")
table_name      = dbutils.widgets.get("table_name")
schema          = dbutils.widgets.get("schema")
dateFormatFile = dbutils.widgets.get("dateFormatFile") or "yyyy-MM-dd"
timestampFormat = (dbutils.widgets.get("timestampFormat") or "yyyy-MM-dd'T'HH:mm:ss[.SSS][XXX]").replace("|",":")
partition_date_column = dbutils.widgets.get("partition_date_column")
original_file_date = dbutils.widgets.get("original_file_date")
starttime_nifi = dbutils.widgets.get("starttime_nifi")
endtime_nifi = dbutils.widgets.get("endtime_nifi")
original_file_size = dbutils.widgets.get("original_file_size")
pipelineRunId = dbutils.widgets.get("pipelineRunId")
catalog_control = dbutils.widgets.get("catalog_control")

In [0]:
esquemas = {
    "schema_det_numero_portados_in": StructType([
        StructField("id_operador",             StringType(),True),      
        StructField("hora_envio_solicitud",         TimestampType(), True),      
        StructField("nt",                StringType(), True),
        StructField("tipo_de_servicio_donante", StringType(), True),
        StructField("tipo_de_servicio_receptor",    StringType(), True),
        StructField("region",    StringType(), True),
        StructField("comuna",    StringType(), True),
        StructField("localidad",    StringType(), True),
        StructField("modalidad",    StringType(), True),
        StructField("rut_del_solicitante",    StringType(), True),
        StructField("id_portabilidad",    StringType(), True),
        StructField("receptor",    IntegerType(), True),
        StructField("donante",    IntegerType(), True),
        StructField("fecha_de_portacion",    TimestampType(), True)
    ]),
    "schema_det_numero_portados_out": StructType([
        StructField("id_operador",              StringType(),True),      
        StructField("hora_envio_solicitud",     TimestampType(), True),      
        StructField("nt",                       StringType(), True),
        StructField("tipo_de_servicio_donante", StringType(), True),
        StructField("tipo_de_servicio_receptor",StringType(), True),
        StructField("region",                   StringType(), True),
        StructField("comuna",                   StringType(), True),
        StructField("localidad",                StringType(), True),
        StructField("modalidad",                StringType(), True),
        StructField("rut_del_solicitante",      StringType(), True),
        StructField("id_portabilidad",          StringType(), True),
        StructField("receptor",                 IntegerType(), True),
        StructField("donante",                  IntegerType(), True),
        StructField("fecha_de_portacion",       TimestampType(), True)
    ]),
    "schema_det_solicitudes_rechazadas": StructType([
        StructField("id_operador",              StringType(),True),      
        StructField("hora_envio_solicitud",     TimestampType(), True),      
        StructField("nt",                       StringType(), True),
        StructField("tipo_de_servicio_donante", StringType(), True),
        StructField("tipo_de_servicio_receptor",StringType(), True),
        StructField("region",                   StringType(), True),
        StructField("comuna",                   StringType(), True),
        StructField("localidad",                StringType(), True),
        StructField("modalidad",                StringType(), True),
        StructField("rut_del_cliente",          StringType(), True),
        StructField("receptor",                 IntegerType(), True),
        StructField("donante",                  IntegerType(), True),
        StructField("codigo_respuesta",         IntegerType(), True),
        StructField("codigo_rechazo",           IntegerType(), True),
        StructField("hora_envio_respuesta",     TimestampType(), True)
    ]),
    "schema_det_solicitudes_portin": StructType([
        StructField("id_operador",              StringType(),True),      
        StructField("hora_envio_solicitud",     TimestampType(), True),      
        StructField("nt",                       StringType(), True),
        StructField("tipo_de_servicio_donante", StringType(), True),
        StructField("tipo_de_servicio_receptor",StringType(), True),
        StructField("region",                   StringType(), True),
        StructField("comuna",                   StringType(), True),
        StructField("localidad",                StringType(), True),
        StructField("canal_de_atencion",        StringType(), True),
        StructField("modalidad",                StringType(), True),
        StructField("rut_del_cliente",          StringType(), True),
        StructField("id_portabilidad",          StringType(), True),
        StructField("receptor",                 IntegerType(), True),
        StructField("donante",                  IntegerType(), True),
        StructField("estado",                   StringType(), True)
    ]),
    "schema_det_solicitudes_portout": StructType([
        StructField("id_operador",              StringType(),True),      
        StructField("hora_envio_solicitud",     TimestampType(), True),      
        StructField("nt",                       StringType(), True),
        StructField("tipo_de_servicio_donante", StringType(), True),
        StructField("tipo_de_servicio_receptor",StringType(), True),
        StructField("region",                   StringType(), True),
        StructField("comuna",                   StringType(), True),
        StructField("localidad",                StringType(), True),
        StructField("canal_de_atencion",        StringType(), True),
        StructField("modalidad",                StringType(), True),
        StructField("rut_del_cliente",          StringType(), True),
        StructField("id_portabilidad",          StringType(), True),
        StructField("receptor",                 IntegerType(), True),
        StructField("donante",                  IntegerType(), True),
        StructField("estado",                   StringType(), True)
    ]),
    "schema_consultas_facti_numero_portin": StructType([
        StructField("id_operador",              StringType(),True),      
        StructField("hora_envio_solicitud",     TimestampType(), True),      
        StructField("id_sgp",                   StringType(), True),
        StructField("codigo_respuesta",         IntegerType(), True),
        StructField("nt",                       StringType(), True),
        StructField("receptor",                 IntegerType(), True),
        StructField("tipo_de_servicio_donante", StringType(), True),
        StructField("tipo_de_servicio_receptor",StringType(), True),
        StructField("portacion_en_curso",       StringType(), True),
        StructField("dias_portacion_prohibida", IntegerType(), True),
        StructField("imei",                     StringType(), True),
        StructField("equipo_bloqueado",         StringType(), True)
    ]),
    "schema_consultas_deuda_portin": StructType([
        StructField("id_operador",              StringType(),True),      
        StructField("sinc_asinc",               StringType(),True),      
        StructField("hora_envio_solicitud",     TimestampType(), True),
        StructField("id_sgp",                   StringType(), True),
        StructField("codigo_respuesta",         IntegerType(), True),     
        StructField("nt",                       StringType(), True),
        StructField("receptor",                 IntegerType(), True),
        StructField("donante",                  IntegerType(), True),
        StructField("tipo_de_servicio_donante", StringType(), True),
        StructField("tipo_de_servicio_receptor",StringType(), True),
        StructField("modalidad",                StringType(), True),
        StructField("rut_del_solicitante",      StringType(), True),
        StructField("estatus_nt",               StringType(), True),
        StructField("ind_titularidad",          StringType(), True),
        StructField("deuda_debida",             LongType(), True),
        StructField("id_doc_factura",           StringType(), True),
        StructField("tipo_servicio_especial",   StringType(), True)
    ]),
    "schema_consultas_deuda_portout": StructType([
        StructField("id_operador",              StringType(),True),      
        StructField("sinc_asinc",               StringType(),True),      
        StructField("hora_envio_resp",          TimestampType(), True),
        StructField("id_sgp",                   StringType(), True),
        StructField("codigo_respuesta",         IntegerType(), True),     
        StructField("nt",                       StringType(), True),
        StructField("receptor",                 IntegerType(), True),
        StructField("donante",                  IntegerType(), True),
        StructField("tipo_de_servicio_donante", StringType(), True),
        StructField("tipo_de_servicio_receptor",StringType(), True),
        StructField("modalidad",                StringType(), True),
        StructField("estatus_nt",               StringType(), True),
        StructField("ind_titularidad",          StringType(), True),
        StructField("deuda_debida",             LongType(), True),
        StructField("id_doc_factura",           StringType(), True),
        StructField("tipo_servicio_especial",   StringType(), True)
    ])
}

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
status = ""
desc_status = ""

schema_selected = esquemas.get(f"schema_{table_name.lower()}", None)

status = "[OK]"
desc_status = "[OK]"
df = spark.read.option("delimiter", delimitador).option("header", encabezado).option("quote", quote).option("encoding", encoding).schema(schema_selected).option("DateFormat", dateFormatFile).option("timestampFormat", timestampFormat).csv(f"{ruta_stage}/{nombre_archivo}")

original_row_count = df.count()

In [0]:
current_id = int(datetime.now().timestamp() * 1000)
formatter_id = datetime.now().strftime("%Y%m%d%H%M%S")
counter = 1
bigdata_ctrl_id = f"{formatter_id}{counter:03d}"

df = df.withColumn("bigdata_close_date",from_unixtime(unix_timestamp(col(partition_date_column), "MM-dd-yyyy"), "yyyy-MM-dd").cast("date")).withColumn("bigdata_ctrl_id", lit(bigdata_ctrl_id).cast("bigint")).withColumn("year", year(col("bigdata_close_date")).cast("string")).withColumn("month", lpad(month(col("bigdata_close_date")).cast("string"), 2, "0")).withColumn("day", lpad(dayofmonth(col("bigdata_close_date")).cast("string"), 2, "0"))
df.show(5, truncate=False)

In [0]:
df_particiones = df.select("bigdata_close_date").distinct()
df_particiones.show()

In [0]:
df_table = spark.read.table(f"{catalogo}.{schema}.{table_name}")
columnas = df_table.columns

df_table_parts = df_table.join(df_particiones, "bigdata_close_date", "inner").select(*[col(c) for c in columnas])

In [0]:
df_full = df.select(*[col(c) for c in columnas]).union(df_table_parts)

In [0]:
columnas_drop_duplicates = [field.name for field in schema_selected.fields]

w_duplicate = Window.partitionBy(columnas_drop_duplicates).orderBy(asc("bigdata_ctrl_id"))
df_sin_duplicados = df_full.withColumn("rn", row_number().over(w_duplicate)).filter("rn = 1").drop("rn")

In [0]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

df_sin_duplicados.write.partitionBy("year","month","day").format("delta").option("partitionOverwriteMode", "dynamic").mode("overwrite").save(f"{dir_adls}/raw")


In [0]:
format_max = "%Y-%m-%dT%H:%M:%S.%fZ"
format_timestamp = "%Y-%m-%dT%H:%M:%SZ"

datetime_fecha_archivo_procesando = datetime.strptime(original_file_date, format_timestamp)
string_fecha_archivo_procesando = datetime_fecha_archivo_procesando.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"  # Tomar solo 3 dígitos de los microsegundos


fecha_actual_str = dbutils.fs.head(f"{dir_adls}/last_ingest_time.txt").split("\n")[1][:-1]
try:
    fecha_actual = datetime.strptime(fecha_actual_str, format_max)
except ValueError:
    fecha_actual = datetime.strptime(fecha_actual_str, format_timestamp)

print(fecha_actual)

if datetime_fecha_archivo_procesando > fecha_actual:
    dbutils.fs.rm(f"{dir_adls}/last_ingest_time.txt", True)
    contents = "timestamp;\n"+string_fecha_archivo_procesando+";"
    dbutils.fs.put(f"{dir_adls}/last_ingest_time.txt", contents, overwrite=True)

In [0]:
dbutils.fs.rm(f"{ruta_stage}/{nombre_archivo}", True)

In [0]:
process_name = table_name
data_source_type = "file"
data_source_name = f"{nombre_archivo}"
endtime_spark = endtime_nifi    
starttime_spark = starttime_nifi
process_type = "normal"
final_file_size = original_file_size

final_row_count = original_row_count
dif_row_count = int(original_row_count) - int(final_row_count)

original_file_size = str(original_file_size)
final_file_size = str(final_file_size)
original_row_count = str(original_row_count)
final_row_count = str(final_row_count)
dif_row_count = str(dif_row_count)

final_number_of_files = str(original_row_count)
end_file_name = table_name
insert_data_ctrl_date = datetime.now().strftime("%Y-%m-%d")
hdfs_path = f"{dir_adls}/raw"

#Formato de Variable con Fechas
original_file_datef = datetime.strptime(original_file_date, "%Y-%m-%dT%H:%M:%SZ")
original_file_date2 = original_file_datef.strftime("%Y-%m-%d %H:%M:%S")

original_file_datef = datetime.strptime(original_file_date, "%Y-%m-%dT%H:%M:%SZ")
original_file_date2 = original_file_datef.strftime("%Y-%m-%d %H:%M:%S")

# Utilizamos substring para extraer las partes de la fecha y hora
anio = starttime_nifi[:4]
mes = starttime_nifi[4:6]
dia = starttime_nifi[6:8]
hora = starttime_nifi[9:11]
minuto = starttime_nifi[11:13]
segundo = starttime_nifi[13:15]
# Formateamos la fecha en el nuevo formato
starttime_nifi2 = f"{anio}-{mes}-{dia} {hora}:{minuto}:{segundo}"

# Utilizamos substring para extraer las partes de la fecha y hora
anio = endtime_nifi[:4]
mes = endtime_nifi[4:6]
dia = endtime_nifi[6:8]
hora = endtime_nifi[9:11]
minuto = endtime_nifi[11:13]
segundo = endtime_nifi[13:15]
endtime_nifi2 = f"{anio}-{mes}-{dia} {hora}:{minuto}:{segundo}"

print("original_file_date2 = " + str(original_file_date2))
print("starttime_nifi2 = " + str(starttime_nifi2))
print("endtime_nifi2 = " + str(endtime_nifi2))

# Convert strings to datetime objects
starttime_nifi_dt = datetime.strptime(starttime_nifi2, "%Y-%m-%d %H:%M:%S")
endtime_nifi_dt = datetime.strptime(endtime_nifi2, "%Y-%m-%d %H:%M:%S")

# Calculate the difference
totaltime_nifi_diff = endtime_nifi_dt - starttime_nifi_dt
print("totaltime_nifi_diff = " + str(totaltime_nifi_diff))
totaltime_nifi_diff_seconds = totaltime_nifi_diff.total_seconds()
print("totaltime_nifi_diff_seconds = " + str(totaltime_nifi_diff_seconds))

totaltime_nifi = str(totaltime_nifi_diff_seconds)

totaltime_spark = float(totaltime_nifi_diff_seconds)
totaltime_process = float(totaltime_nifi_diff_seconds)

print("totaltime_spark = " + str(totaltime_spark))
print("totaltime_process = " + str(totaltime_process))

print("bigdata_ctrl_id " + str(bigdata_ctrl_id))
print("process_name " + str(process_name))
print("data_source_type " + str(data_source_type))
print("data_source_name " + str(data_source_name))
print("original_file_date " + str(original_file_date))
print("starttime_nifi " + str(starttime_nifi))
print("endtime_nifi " + str(endtime_nifi))
print("totaltime_nifi " + str(totaltime_nifi))
print("starttime_spark " + str(starttime_spark))
print("endtime_spark " + str(endtime_spark))
print("totaltime_spark " + str(totaltime_spark))
print("totaltime_process " + str(totaltime_process))
print("insert_data_ctrl_date " + str(insert_data_ctrl_date))
print("process_type " + str(process_type))
print("original_file_size " + str(original_file_size))
print("final_file_size " + str(final_file_size))
print("original_row_count " + str(original_row_count))
print("final_row_count " + str(final_row_count))
print("dif_row_count " + str(dif_row_count))
print("final_number_of_files " + str(final_number_of_files))
print("end_file_name " + str(end_file_name))
print("hdfs_path " + str(hdfs_path))
print("pipelineRunId " + str(pipelineRunId))
print("status " + str(status))
print("desc_status " + str(desc_status))
print("catalog_control" + str(catalog_control))

data_control = [(bigdata_ctrl_id, process_name, data_source_type, data_source_name, original_file_date2,
starttime_nifi2, endtime_nifi2, totaltime_nifi, starttime_spark, endtime_spark ,
totaltime_spark, totaltime_process, insert_data_ctrl_date, process_type, 
original_file_size, final_file_size, original_row_count, final_row_count, 
dif_row_count, final_number_of_files, end_file_name, hdfs_path,
pipelineRunId, status, desc_status,  bigdata_ctrl_id )]

columns = ["big_data_ctrl_id", "process_name", "data_source_type", "data_source_name", "original_file_date",
"starttime_nifi", "endtime_nifi", "totaltime_nifi", "starttime_spark", "endtime_spark", 
"totaltime_spark", "totaltime_process", "insert_data_ctrl_date", "process_type", 
"original_file_size", "final_file_size", "original_row_count", "final_row_count", 
"dif_row_count", "final_number_of_files", "end_file_name", "hdfs_path", 
"pipelineRunId", "status", "desc_status", "bigdata_ctrl_id"]

df_data_control = spark.createDataFrame(data_control, columns)

df_data_control = df_data_control \
.withColumn("original_file_date", from_unixtime(unix_timestamp(col("original_file_date"), "yyyy-MM-dd HH:mm:ss"), "yyyy-MM-dd HH:mm:ss")) \
.withColumn("starttime_nifi", from_unixtime(unix_timestamp(col("starttime_nifi"), "yyyy-MM-dd HH:mm:ss"), "yyyy-MM-dd HH:mm:ss")) \
.withColumn("endtime_nifi", from_unixtime(unix_timestamp(col("endtime_nifi"), "yyyy-MM-dd HH:mm:ss"), "yyyy-MM-dd HH:mm:ss")) \
.withColumn("totaltime_spark", col("totaltime_spark").cast("float")) \
.withColumn("totaltime_process", col("totaltime_process").cast("float")) \
.withColumn("big_data_ctrl_id", col("big_data_ctrl_id").cast("string")) \
.withColumn("bigdata_ctrl_id", col("bigdata_ctrl_id").cast("string"))

df_data_control.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(catalog_control)
